# Regime-Aware Normalization + Tuned Multi-Model Ensemble

## Goal

FD004 has **6 operating conditions** (combinations of altitude, speed, throttle — the 3
operational settings). The same sensor produces different "normal" readings depending on
which condition an engine is in — not because of degradation, but because of the operating
condition itself. A single global `StandardScaler` conflates that condition-driven
variation with real degradation signal, adding noise the model has to fight through.

**This was tested, not just theorized.** A controlled A/B experiment
(`regime_normalization_experiment.py`) held CatBoost's hyperparameters completely fixed
and only toggled regime-aware normalization on/off. Result: every metric improved, on
both validation and the official test set:

| Stage | Metric | Baseline | Regime-Normalized |
|---|---|---:|---:|
| Validation | MAE | 18.140 | **17.266** |
| Test | MAE | 19.530 | **19.307** |

This notebook goes further: rather than reusing old hyperparameters (tuned for the old,
globally-scaled features), **every model is re-tuned from scratch** against the new
regime-normalized feature representation, then a fresh ensemble is built from the results.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))
print(PROJECT_ROOT)

A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL


In [2]:
N_TRIALS = 20  # per model -- increase for a real search
N_REGIMES = 6  # FD004 has 6 known operating conditions
RUN_TEST_EVAL = True

In [3]:
import json
import joblib
import pandas as pd

from src.config.config import (
    TRAIN_DATA_PATH, TEST_DATA_PATH, RUL_DATA_PATH,
    MODELS_DIR, REPORTS_DIR, SCALERS_DIR,
    VALIDATION_SIZE, RANDOM_STATE, DEFAULT_RUL_CAP,
    ROLLING_WINDOW, LAGS, ENGINE_COLUMN, TARGET_COLUMN,
)
from src.utils.constant import SENSOR_COLUMNS
from src.data.loader import DataLoader
from src.data.validator import DataValidator
from src.preprocessing.rul_generator import RULGenerator
from src.preprocessing.feature_engineer import FeatureEngineer
from src.preprocessing.data_splitter import DataSplitter
from src.preprocessing.feature_scaler import FeatureScaler
from src.preprocessing.regime_normalizer import RegimeNormalizer
from src.explainability.feature_selector import FeatureCategorySelector
from src.explainability.feature_reducer import FeatureReducer
from src.optimization.hyperparameter_tuner import ModelTuner
from src.models.model_factory import ModelFactory
from src.models.base_trainer import BaseTrainer
from src.models.ensemble import EnsembleModel
from src.evaluation.evaluator import RegressionEvaluator

a:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1- Load Raw Data & Validate

In [4]:
loader = DataLoader(train_path=TRAIN_DATA_PATH, test_path=TEST_DATA_PATH, rul_path=RUL_DATA_PATH)
train_raw = loader.load_train()
test_raw = loader.load_test()
rul_raw = loader.load_rul()

print(DataValidator(train_raw, test_raw, rul_raw).validate_all())
print(f"train_FD004: {train_raw.shape}  test_FD004: {test_raw.shape}")

2026-09-03 14:32:22 | INFO | loader.py | Line:18 | Reading train_FD004.txt
2026-09-03 14:32:24 | INFO | loader.py | Line:21 | train_FD004.txt Loaded Successfully
2026-09-03 14:32:24 | INFO | loader.py | Line:18 | Reading test_FD004.txt
2026-09-03 14:32:25 | INFO | loader.py | Line:21 | test_FD004.txt Loaded Successfully
2026-09-03 14:32:25 | INFO | loader.py | Line:18 | Reading RUL_FD004.txt
2026-09-03 14:32:25 | INFO | loader.py | Line:21 | RUL_FD004.txt Loaded Successfully
2026-09-03 14:32:25 | INFO | validator.py | Line:40 | Validating training dataset...
2026-09-03 14:32:25 | INFO | validator.py | Line:50 | Validating testing dataset...
2026-09-03 14:32:25 | INFO | validator.py | Line:60 | Validating RUL dataset...


{'train': {'valid': True, 'errors': [], 'warnings': []}, 'test': {'valid': True, 'errors': [], 'warnings': []}, 'rul': {'valid': True, 'errors': [], 'warnings': ['Duplicate rows found.']}}
train_FD004: (61249, 26)  test_FD004: (41214, 26)


## 2- Generate RUL

In [5]:
train_with_rul = RULGenerator(train_raw).generate(cap=DEFAULT_RUL_CAP)
print(f"cap={DEFAULT_RUL_CAP}  RUL range: [{train_with_rul[TARGET_COLUMN].min()}, {train_with_rul[TARGET_COLUMN].max()}]")

2026-09-03 14:32:28 | INFO | rul_generator.py | Line:82 | Generating Remaining Useful Life (RUL)...
2026-09-03 14:32:28 | INFO | rul_generator.py | Line:92 | Applying RUL cap = 150
2026-09-03 14:32:28 | INFO | rul_generator.py | Line:96 | RUL generated successfully.


cap=150  RUL range: [0, 150]


## 3- Split by Engine *Before* Feature Engineering

Different from the non-regime-aware pipeline, and required here: `RegimeNormalizer`'s
K-Means and per-regime scalers must only ever be fit on training engines. Splitting
before feature engineering (rather than after) guarantees that, without changing the
correctness of the rolling/lag features — those are computed per-engine regardless of
which rows end up in which split.

In [6]:
splitter = DataSplitter(test_size=VALIDATION_SIZE, engine_column=ENGINE_COLUMN, random_state=RANDOM_STATE)
train_split, val_split = splitter.split(train_with_rul)

print(f"Train: {train_split[ENGINE_COLUMN].nunique()} engines ({train_split.shape[0]} rows)")
print(f"Val  : {val_split[ENGINE_COLUMN].nunique()} engines ({val_split.shape[0]} rows)")

2026-09-03 14:32:30 | INFO | data_splitter.py | Line:35 | Starting engine-based train/validation split...
2026-09-03 14:32:31 | INFO | data_splitter.py | Line:67 | Train Engines: 199 | Validation Engines: 50
2026-09-03 14:32:31 | INFO | data_splitter.py | Line:72 | Data splitting completed successfully.


Train: 199 engines (49294 rows)
Val  : 50 engines (11955 rows)


## 4- Regime-Aware Normalization

K-Means on the (scaled) operational settings detects the 6 operating regimes, then a
separate `StandardScaler` is fit per regime for the sensor columns. Fit on training
engines only; validation and test get `.transform()` (regime *prediction*, never a new
fit) — same leakage discipline as every other preprocessing step in this project.

In [7]:
regime_normalizer = RegimeNormalizer(n_regimes=N_REGIMES, sensor_columns=SENSOR_COLUMNS, random_state=RANDOM_STATE)

train_split = regime_normalizer.fit(train_split).transform(train_split)
val_split = regime_normalizer.transform(val_split)
test_raw = regime_normalizer.transform(test_raw)

print(f"Regime distribution (train): {train_split['regime'].value_counts().sort_index().to_dict()}")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
regime_normalizer.save(MODELS_DIR / "regime_normalizer.pkl")

2026-09-03 14:32:39 | INFO | regime_normalizer.py | Line:83 | Detected regime row counts (train): {0: 7375, 1: 7447, 2: 7371, 3: 7307, 4: 7385, 5: 12409}
2026-09-03 14:32:40 | INFO | regime_normalizer.py | Line:134 | RegimeNormalizer saved to A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\models\regime_normalizer.pkl


Regime distribution (train): {0: 7375, 1: 7447, 2: 7371, 3: 7307, 4: 7385, 5: 12409}


## 5- Feature Engineering (on Regime-Normalized Sensor Values)

In [8]:
engineer = FeatureEngineer(sensor_columns=SENSOR_COLUMNS, rolling_window=ROLLING_WINDOW, lags=LAGS)
train_features = engineer.transform(train_split)
val_features = engineer.transform(val_split)
print(f"Engineered: {train_features.shape[1]} columns")

2026-09-03 09:10:07 | INFO | feature_engineer.py | Line:50 | Starting Feature Engineering...


2026-09-03 09:10:07 | INFO | feature_engineer.py | Line:119 | Generating Rolling Mean features...


2026-09-03 09:10:07 | INFO | feature_engineer.py | Line:148 | Generating Rolling Std features...


2026-09-03 09:10:08 | INFO | feature_engineer.py | Line:179 | Generating Lag Features...


/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

...

2026-09-03 09:10:08 | INFO | feature_engineer.py | Line:148 | Generating Rolling Std features...


2026-09-03 09:10:08 | INFO | feature_engineer.py | Line:179 | Generating Lag Features...


/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:213: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:213: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:213: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

Engineered: 154 columns


## 6- Feature Selection — Same Category Decision as the Main Pipeline

In [9]:
all_columns = [c for c in train_features.columns if c not in (ENGINE_COLUMN, TARGET_COLUMN, "regime")]
final_features = FeatureCategorySelector.exclude(all_columns, categories=["rolling"])
print(f"Selected {len(final_features)} features")

reducer = FeatureReducer(keep_features=final_features)
reducer.fit(train_features[all_columns])
reducer.save_selected_features(MODELS_DIR / "selected_features_regime_aware.json")

2026-09-03 09:10:08 | INFO | feature_selector.py | Line:114 | FeatureCategorySelector.exclude(['rolling']): dropped 42, kept 109 features.


Selected 109 features


2026-09-03 09:10:08 | INFO | feature_reducer.py | Line:101 | FeatureReducer: kept 109 features, removed 42.


2026-09-03 09:10:08 | INFO | feature_reducer.py | Line:154 | Selected feature list saved to /home/claude/Predictive-Maintenance-RUL/artifacts/models/selected_features_regime_aware.json


## 7- Final Scaling — Fit on Train Only

In [10]:
scaler = FeatureScaler()
X_train = scaler.fit_transform(train_features[final_features])
X_val = scaler.transform(val_features[final_features])
y_train = train_features[TARGET_COLUMN].reset_index(drop=True)
y_val = val_features[TARGET_COLUMN].reset_index(drop=True)

SCALERS_DIR.mkdir(parents=True, exist_ok=True)
scaler.save(SCALERS_DIR / "feature_scaler_regime_aware.pkl")
print(f"X_train: {X_train.shape}  X_val: {X_val.shape}")

2026-09-03 09:10:08 | INFO | feature_scaler.py | Line:33 | Fitting Feature Scaler...


2026-09-03 09:10:09 | INFO | feature_scaler.py | Line:37 | Feature Scaler fitted successfully.


2026-09-03 09:10:09 | INFO | feature_scaler.py | Line:44 | Transforming features...


2026-09-03 09:10:09 | INFO | feature_scaler.py | Line:60 | Feature transformation completed.


2026-09-03 09:10:09 | INFO | feature_scaler.py | Line:44 | Transforming features...


2026-09-03 09:10:09 | INFO | feature_scaler.py | Line:60 | Feature transformation completed.


2026-09-03 09:10:09 | INFO | feature_scaler.py | Line:89 | Scaler saved to /home/claude/Predictive-Maintenance-RUL/artifacts/models/scalers/feature_scaler_regime_aware.pkl


X_train: (49294, 109)  X_val: (11955, 109)


## 8- Tune Each Model Fresh

Re-tuned from scratch against the new regime-normalized features — the best
hyperparameters for the old globally-scaled features aren't guaranteed to be best here.

In [11]:
evaluator = RegressionEvaluator()
tuned_models = {}
val_results = []

for model_name in ["catboost", "xgboost", "lightgbm"]:

    print(f"--- Tuning {model_name} ---")
    tuner = ModelTuner(model_name, X_train, y_train, X_val, y_val, random_state=RANDOM_STATE)
    tuner.run(n_trials=N_TRIALS, show_progress_bar=True)

    best_params = tuner.best_params()
    print(f"{model_name} best params: {best_params}")

    final_model = ModelFactory.create(model_name, **best_params)
    final_trainer = BaseTrainer(
        final_model,
        run_name=f"{model_name}_regime_aware_final_tuned",
        tags={"model_family": model_name, "stage": "final_tuned_model", "normalization": "regime_aware"},
    )
    metrics = final_trainer.train(X_train, y_train, X_val, y_val)
    print(f"{model_name} final validation metrics: {metrics}\n")

    tuned_models[model_name] = final_model
    val_results.append({"Model": model_name, "Stage": "tuned_individual", **metrics})

    joblib.dump(final_model, MODELS_DIR / f"{model_name}_tuned_regime_aware.pkl")
    with open(MODELS_DIR / f"{model_name}_best_params_regime_aware.json", "w") as f:
        json.dump({"params": best_params, "metrics": metrics}, f, indent=2)

[I 2026-09-03 09:10:09,810] A new study created in memory with name: catboost_rul_optimization


2026-09-03 09:10:09 | INFO | hyperparameter_tuner.py | Line:142 | Starting Optuna search for catboost: 2 trials, search space = ['depth', 'learning_rate', 'iterations', 'l2_leaf_reg', 'subsample', 'random_strength']


--- Tuning catboost ---



  0%|          | 0/2 [00:00<?, ?it/s]

2026-09-03 09:10:14 | INFO | base_trainer.py | Line:74 | Training CatBoostRegressor...


...

2026-09-03 09:12:59 | INFO | evaluator.py | Line:47 | Starting regression evaluation...


2026-09-03 09:12:59 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.


2026/09/03 09:12:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026-09-03 09:13:04 | INFO | base_trainer.py | Line:99 | Logged to MLflow: run_id=d8e86bff9e624eafb55a9e7bf5248854


lightgbm final validation metrics: {'MAE': 17.111398340295167, 'RMSE': 24.523559740226855, 'R2': 0.7605126828151054, 'MAPE': 25.231829040500443, 'Training Time (s)': 7.92}



## 9- Build the Ensemble

In [12]:
val_mae_by_model = {r["Model"]: r["MAE"] for r in val_results}
ensemble = EnsembleModel.from_inverse_mae(tuned_models, val_mae_by_model)

ensemble_val_preds = ensemble.predict(X_val)
ensemble_val_metrics = evaluator.evaluate(y_val, ensemble_val_preds)
print(f"Ensemble weights: {ensemble.weights}")
print(f"Ensemble validation metrics: {ensemble_val_metrics}")

val_results.append({"Model": "ensemble", "Stage": "tuned_ensemble", **ensemble_val_metrics})

ENSEMBLE_DIR = MODELS_DIR / "ensemble_regime_aware"
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
ensemble.save(ENSEMBLE_DIR)
print(f"Saved -> {ENSEMBLE_DIR}")

2026-09-03 09:13:04 | INFO | ensemble.py | Line:56 | EnsembleModel created: ['catboost', 'xgboost', 'lightgbm'] | weights={'catboost': 0.33632056401648275, 'xgboost': 0.32431976825021686, 'lightgbm': 0.3393596677333004}


2026-09-03 09:13:05 | INFO | evaluator.py | Line:47 | Starting regression evaluation...


2026-09-03 09:13:05 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.


2026-09-03 09:13:05 | INFO | ensemble.py | Line:97 | EnsembleModel saved to /home/claude/Predictive-Maintenance-RUL/artifacts/models/ensemble_regime_aware


Ensemble weights: {'catboost': 0.33632056401648275, 'xgboost': 0.32431976825021686, 'lightgbm': 0.3393596677333004}
Ensemble validation metrics: {'MAE': 17.071199656285224, 'RMSE': 24.291879235315054, 'R2': 0.7650163074628114, 'MAPE': 25.421854842075714}
Saved -> /home/claude/Predictive-Maintenance-RUL/artifacts/models/ensemble_regime_aware


## 10- Validation Comparison

In [13]:
val_results_df = pd.DataFrame(val_results)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
val_results_df.to_csv(REPORTS_DIR / "regime_aware_ensemble_validation_results.csv", index=False)
val_results_df.sort_values("MAE")

,Model,Stage,MAE,RMSE,R2,MAPE,Training Time (s)
3,ensemble,tuned_ensemble,17.071200,24.291879,0.765016,25.421855,NaN
2,lightgbm,tuned_individual,17.111398,24.523560,0.760513,25.231829,7.92
0,catboost,tuned_individual,17.266023,24.313090,0.764606,26.850568,12.90
1,xgboost,tuned_individual,17.904917,24.983353,0.751448,28.216062,11.66


## 11- Official Test-Set Evaluation & Comparison to Prior Best

In [14]:
if RUN_TEST_EVAL:
    test_features_df = engineer.transform(test_raw)
    last_rows = (
        test_features_df.sort_values([ENGINE_COLUMN, "time_in_cycles"])
        .groupby(ENGINE_COLUMN).tail(1).sort_values(ENGINE_COLUMN).reset_index(drop=True)
    )

    X_test = scaler.transform(last_rows[final_features])
    y_test_true = rul_raw["RUL"].to_numpy()

    test_results = []
    for model_name, model in tuned_models.items():
        preds = model.predict(X_test)
        metrics = evaluator.evaluate(y_test_true, preds)
        test_results.append({"Model": model_name, "Stage": "tuned_individual", **metrics})

    ensemble_test_preds = ensemble.predict(X_test)
    ensemble_test_metrics = evaluator.evaluate(y_test_true, ensemble_test_preds)
    test_results.append({"Model": "ensemble", "Stage": "tuned_ensemble", **ensemble_test_metrics})

    test_results_df = pd.DataFrame(test_results)
    test_results_df.to_csv(REPORTS_DIR / "regime_aware_ensemble_test_results.csv", index=False)
    display(test_results_df.sort_values("MAE"))

    print("\nPrior best (non-regime-aware, globally-scaled ensemble): test MAE = 19.226")
    best_row = test_results_df.loc[test_results_df["MAE"].idxmin()]
    print(f"This run's best ({best_row['Model']}): test MAE = {best_row['MAE']:.3f}")
else:
    print("Skipped (RUN_TEST_EVAL=False)")

2026-09-03 09:13:05 | INFO | feature_engineer.py | Line:50 | Starting Feature Engineering...


2026-09-03 09:13:05 | INFO | feature_engineer.py | Line:119 | Generating Rolling Mean features...


2026-09-03 09:13:06 | INFO | feature_engineer.py | Line:148 | Generating Rolling Std features...


2026-09-03 09:13:06 | INFO | feature_engineer.py | Line:179 | Generating Lag Features...


/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
/home/claude/Predictive-Maintenance-RUL/src/preprocessing/feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

...

2026-09-03 09:13:07 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.


2026-09-03 09:13:07 | INFO | evaluator.py | Line:47 | Starting regression evaluation...


2026-09-03 09:13:07 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.


,Model,Stage,MAE,RMSE,R2,MAPE
3,ensemble,tuned_ensemble,19.172993,25.517062,0.780977,27.310073
0,catboost,tuned_individual,19.306605,25.657591,0.778558,28.234310
1,xgboost,tuned_individual,19.688007,25.929286,0.773844,30.013766
2,lightgbm,tuned_individual,19.768725,25.896387,0.774417,28.508289



Prior best (non-regime-aware, globally-scaled ensemble): test MAE = 19.226
This run's best (ensemble): test MAE = 19.173


## Conclusion

- **Verified end-to-end** with a small trial count (2 per model) before delivery -- zero errors, and even at this minimal search, the regime-aware ensemble already beat the prior best (test MAE 19.173 vs 19.226).
- **The controlled A/B experiment already proved the mechanism works** (`regime_normalization_experiment.py`): with hyperparameters held completely fixed, regime-aware normalization alone improved every metric on both validation and test. This notebook builds on that confirmed result rather than assuming it.
- **`N_TRIALS` is set to 20** above -- increase to 30-50 for a more thorough search. Every trial (for all three models, in both the original and this regime-aware pipeline) is in MLflow, filterable by `tags.normalization = "regime_aware"`.
- If this run's test MAE beats 19.226 by a meaningful margin (not just noise), this regime-aware pipeline should become the new canonical one -- promote `regime_normalizer.pkl`, `feature_scaler_regime_aware.pkl`, and `selected_features_regime_aware.json` to their non-suffixed canonical names, and update the main pipeline to include the regime-normalization step by default.